In [ ]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
import sys
sys.path.append("../")
import models
from utils.training import train_meta_model, train_gp
from utils.gp_data import obtain_me_a_nice_gp_dataset_please
from utils.data_utils import ctxt_trgt_split, obtain_me_a_nice_sawtooth_dataset_please, obtain_me_a_nice_heaviside_dataset_please
from networks.base_architectures import Sin

torch.set_default_dtype(torch.float64)

%load_ext autoreload
%autoreload 2

### Make Dataset

In [ ]:
num_datasets = 200
md = []
# gp_data_hypers = {'l': 1.0, 'kernel': 'per', 'p': 1, 'x_range': [-5.0, 5.0]}
gp_data_hypers = {'l': 0.5, 'kernel': 'se', 'x_range': [-5.0, 5.0]}
# st_data_hypers = {'p': 1.0, 'random_shift': False, 'random_gradient': False, 'x_range': [-5.0, 5.0]}
# h_data_hypers = {'x_range': [-5.0, 5.0], 'l': 1}
for _ in range(num_datasets):
    X, y = obtain_me_a_nice_gp_dataset_please(n_range=[40, 100], **gp_data_hypers)
    # X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[40, 100], **st_data_hypers)
    # X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[40, 100], **h_data_hypers)
    md.append((X, y))

### Initialise Model

In [ ]:
lik = models.GaussianLikelihood(y_dim=1, sigma_y=0.05, train=False, sigma_y_upper_bound=0.2)
bdnp = models.BDNP(x_dim=1,
                   y_dim=1,
                   hidden_dims=[50, 50],
                   prior_type=1,
                   likelihood=lik,
                   inf_dims=[50, 50], 
                   use_final_layer_targets=True,
                   use_final_layer_noise=True,
                   scale_prior=True,
                   nonlinearity=Sin(),
                   )

### Train the Model

In [ ]:
# training_metrics = train_meta_model(
#     bdnp,
#     md,
#     training_steps=10_000,
#     batch_size=5,
#     learning_rate=5e-3,
#     final_learning_rate=1e-4,
#     num_samples=1,
#     loss_function='npvi',
#     # ctxt_proportion_range=(0.6, 0.9),
#     release_prior_at_step=250,
#     anneal_ctxt_proportion=True,
#     ctxt_anneal_start_step=2000,
#     ctxt_anneal_start_proportion=1.0,
#     ctxt_anneal_end_step=8000,
#     ctxt_anneal_end_proportion=0.75,
# )

training_metrics = train_meta_model(
    bdnp,
    md,
    training_steps=2_000,
    batch_size=5,
    learning_rate=5e-3,
    num_samples=1,
    loss_function='vi',
    use_gpu=True,
)

In [ ]:
fig, axes = plt.subplots(1, len(training_metrics), figsize=(3*len(training_metrics), 1))
omitted_steps = 0
for i, (key, value) in enumerate(training_metrics.items()):
    axes[i].plot(value[omitted_steps:])
    axes[i].set_xlabel(key)
    axes[i].grid()
    # axes[i].set_ylim([-100, 400])
plt.show()


### Visualise test predictions

In [ ]:
prior_samps = False
prior_samps = True

if prior_samps:
    X, y = obtain_me_a_nice_gp_dataset_please(n_range=[1, 2], **gp_data_hypers)
    # X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[1, 2], **st_data_hypers)
    # X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[1, 4], **h_data_hypers)
    X_c, y_c = X.clone(), y.clone()
    X_c, y_c = None, None
    samps = 100
else:
    # X, y = obtain_me_a_nice_gp_dataset_please(n_range=[10, 100], **gp_data_hypers)
    X, y = obtain_me_a_nice_gp_dataset_please(n_range=[10, 20], **gp_data_hypers)
    # X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[10, 100], **st_data_hypers)
    # X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[10, 100], **h_data_hypers)
    # X, y = md[0]
    X_c, y_c = X.clone(), y.clone()
    samps = 100


xs = torch.linspace(-5.0, 5.0, 200).unsqueeze(-1)
with torch.no_grad():
    pred_samps = bdnp(xs, X_c, y_c, num_samples=samps, update_prev=True, save_stuff=False)[0]

plt.plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
if X_c is not None:
    plt.scatter(X_c, y_c, color='C1', zorder=10000)
plt.grid()
plt.xlim([-5.0, 5.0])
plt.ylim([-5.0, 5.0])
plt.show()

### Online Learning Demo

In [ ]:
X, y = obtain_me_a_nice_gp_dataset_please(n_range=[30, 31], **gp_data_hypers)
# X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[30, 32], **st_data_hypers)
# X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[30, 32], **h_data_hypers)
X_c, y_c = X.clone(), y.clone()
samps = 100
inds = [0, 3, 8, 15, 30]

fig, axes = plt.subplots(4, 3, sharex=True, sharey=True, figsize=(9, 12))

xs = torch.linspace(-5.0, 5.0, 200).unsqueeze(-1)

for j in range(1, len(inds)):

    with torch.no_grad():
        full_pred_samps = bdnp(xs, X_c[:inds[j]], y_c[:inds[j]], num_samples=samps)[0]
        seq_pred_samps = bdnp(xs, X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], num_samples=samps, update_prev=(j!=1), save_stuff=True)[0]
        control_pred_samps = bdnp(xs, X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], num_samples=samps)[0]

    axes[j-1][0].plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, full_pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
    axes[j-1][0].scatter(X_c[:inds[j]], y_c[:inds[j]], color='C1', zorder=1000)

    axes[j-1][1].plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, seq_pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
    axes[j-1][1].scatter(X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], color='C1', zorder=1000)
    axes[j-1][1].scatter(X_c[:inds[j-1]], y_c[:inds[j-1]], color='black', zorder=1000)

    axes[j-1][2].plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, control_pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
    axes[j-1][2].scatter(X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], color='C1', zorder=1000)

    for k in range(3):
        axes[j-1][k].grid()
        axes[j-1][k].set_xlim([-5.0, 5.0])
        axes[j-1][k].set_ylim([-5.0, 5.0])

plt.show()